In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision import models
from PIL import Image

In [2]:
train=r"E:\engg\Data\Tumor_data\Train"
test=r"E:\engg\Data\Tumor_data\Test"
val=r"E:\engg\Data\Tumor_data\Val"

In [3]:
transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

In [4]:
train_data = datasets.ImageFolder(root=train,transform=transform_pipeline)
test_data = datasets.ImageFolder(root=test,transform=transform_pipeline)
val_data = datasets.ImageFolder(root=test,transform=transform_pipeline)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
val_loader = DataLoader(val_data, batch_size=32, shuffle=True)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for	para in model.parameters():
    para.requires_grad=False		
model.fc = nn.Linear(model.fc.in_features, 4)
model = model.to(device)

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
epochs = 10
best_val_loss = float("inf")
patience = 5
counter = 0

for epoch in range(epochs):
	model.train()
	running_loss=0.0
	for	imgs,labels	in train_loader:
		imgs,labels = imgs.to(device), labels.to(device)
		optimizer.zero_grad()
		outputs= model(imgs)
		loss=criterion(outputs,	labels)	
		loss.backward()	
		optimizer.step()
		running_loss += loss.item()
	train_loss = running_loss / len(train_loader)
	model.eval()
	val_loss = 0.0
	correct = 0
	total = 0      
	with torch.no_grad():
		for imgs, labels in val_loader:
			imgs, labels = imgs.to(device), labels.to(device)
			outputs = model(imgs)
			loss = criterion(outputs, labels)
			val_loss += loss.item()
			preds = outputs.argmax(dim=1)
			correct += (preds == labels).sum().item()
			total += labels.size(0)
	val_loss = val_loss / len(val_loader)
	val_accuracy = correct / total
	print(f"Epoch {epoch+1}/{epochs} |train loss:{train_loss:.4f} |val loss:{val_loss:.4f} |val accuracy:{val_accuracy:.4f}")


Epoch 1/10 |train loss:0.3116 |val loss:0.5135 |val accuracy:0.8287
Epoch 2/10 |train loss:0.2938 |val loss:0.4745 |val accuracy:0.8425
Epoch 3/10 |train loss:0.2909 |val loss:0.5042 |val accuracy:0.8419
Epoch 4/10 |train loss:0.2847 |val loss:0.4984 |val accuracy:0.8344
Epoch 5/10 |train loss:0.2808 |val loss:0.5024 |val accuracy:0.8475
Epoch 6/10 |train loss:0.2700 |val loss:0.4800 |val accuracy:0.8519
Epoch 7/10 |train loss:0.2454 |val loss:0.4492 |val accuracy:0.8638
Epoch 8/10 |train loss:0.2629 |val loss:0.5129 |val accuracy:0.8556
Epoch 9/10 |train loss:0.2465 |val loss:0.4363 |val accuracy:0.8638
Epoch 10/10 |train loss:0.2407 |val loss:0.5097 |val accuracy:0.8544


In [11]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predicted = outputs.argmax(dim=1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 85.44%


In [12]:
torch.save(model.state_dict(), "brain_tumor_resnet.pth")